# 02 · Process — Clean, validate, and build the analysis tables

Decisions come from [reports/02_prepare.md](../reports/02_prepare.md). The full write-up is [reports/03_process.md](../reports/03_process.md). The audit of Google's lab uses the lab's own table and lives in [notebook 03](03_analyze_lab_audit.ipynb).

```
ga_sessions_*  →  sql/process/01_sessions_clean.sql  →  sessions_clean.parquet
                        ├── remarketing_table  (D2 model: corrected)
                        └── touches / journeys (D1 attribution)
```

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from src import bq
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
from src.validate import validate_sessions
from src.tables import remarketing_table
from src.journeys import touches, journeys, revenue_cap

## 1. Clean session table

One row per session. It scans 0.81 GB the first time (about 8 minutes over REST) and reads from the cache after that. The SQL documents every cleaning rule.

In [2]:
sessions = bq.query("process/01_sessions_clean.sql", cache="sessions_clean")
sessions.shape

(903653, 32)

In [3]:
sessions.sort_values("session_key").head(3).T

,418433,29961,782531
session_key,0000010278554503158-1477029466-1477029466,0000020424342248747-1480578901-1480578901,0000027376579751715-1486866293-1486866293
full_visitor_id,0000010278554503158,0000020424342248747,0000027376579751715
visit_id,1477029466,1480578901,1486866293
visit_start_time,1477029466,1480578901,1486866293
session_start,2016-10-21 05:57:46+00:00,2016-12-01 07:55:01+00:00,2017-02-12 02:24:53+00:00
session_date,2016-10-20 00:00:00,2016-11-30 00:00:00,2017-02-11 00:00:00
visit_number,1,1,1
is_new_visit,True,True,True
channel,Organic Search,Organic Search,Organic Search
source,google,(direct),(direct)


## 2. Validation against raw totals

Five checks compare the extract with totals measured on the raw BigQuery tables in Prepare (rows, purchase sessions, revenue and the internal segment). The others are internal-consistency checks.

In [4]:
checks = validate_sessions(sessions)
assert (checks.status == 'PASS').all(), checks[checks.status != 'PASS']
checks

,status,check,observed
0,PASS,Row count matches raw,"903,653"
1,PASS,session_key is unique,"903,653 unique"
2,PASS,Purchase sessions match raw,"11,552"
3,PASS,Revenue matches raw,"$1,780,149"
4,PASS,Internal visitors match Prepare,"37,332"
5,PASS,Internal sessions / purchases / revenue match ...,"76,536 / 5,408 / $730,376.50"
6,PASS,Internal flag is constant within each visitor,
7,PASS,No NULLs in engagement / outcome columns,
8,PASS,No negative values,
9,PASS,Date range is 2016-08-01 to 2017-08-01,2016-08-01 to 2017-08-01


In [5]:
internal = sessions.loc[sessions.is_internal, "revenue_usd"].sum()
store = sessions.revenue_usd.sum()
p02 = bq.query("prepare/p02_internal_segment_size.sql", cache="p02_internal_segment_size").set_index("segment").revenue_usd
p02_internal = p02["Internal (likely employees)"]
pd.Series({
    "internal revenue, exact": f"${internal:,.2f}",
    "  rounded by BigQuery (half away from zero; Prepare query p02)": f"${p02_internal:,.0f}",
    "  rounded by Python (half to even)": f"${round(internal):,.0f}",
    "  difference": f"${p02_internal - round(internal):,.0f}",
    "store revenue, exact": f"${store:,.2f}",
    "  sum of p02's two rounded segment rows": f"${p02.sum():,.0f}",
    "internal share of store revenue": f"{internal / store:.0%} ({internal / store:.2%})",
})

internal revenue, exact                                             $730,376.50
  rounded by BigQuery (half away from zero; Prepare query p02)         $730,377
  rounded by Python (half to even)                                     $730,376
  difference                                                                 $1
store revenue, exact                                              $1,780,149.23
  sum of p02's two rounded segment rows                              $1,780,150
internal share of store revenue                                    41% (41.03%)
dtype: str

## 3. GA already shifts credit to earlier campaigns

When a visitor returns directly, GA360 labels the visit with their *previous* campaign and sets `isTrueDirect`. GA also sets the flag when two visits in a row carry identical campaign details, such as a repeat Google search. The export can't tell these apart, so the relabel below reads every flagged visit as a direct return, and Analyze ([Part C, §7](../reports/04_analyze.md)) treats it as the top of a range.

In [6]:
ext = sessions[~sessions.is_internal]
relabelled = ext[ext.is_true_direct & (ext.channel != "Direct")]
pd.Series({
    "external sessions": len(ext),
    "non-Direct label but actually a direct return": len(relabelled),
    "  share of external sessions": f"{len(relabelled)/len(ext):.1%}",
    "external purchases": int(ext.purchased.sum()),
    "  purchases on relabelled sessions": int(relabelled.purchased.sum()),
    "  share": f"{relabelled.purchased.sum()/ext.purchased.sum():.1%}",
})

external sessions                                827117
non-Direct label but actually a direct return     96809
  share of external sessions                      11.7%
external purchases                                 6144
  purchases on relabelled sessions                 1790
  share                                           29.1%
dtype: object

In [7]:
print(f"{(relabelled.visit_number > 1).sum():,} of {len(relabelled):,} relabeled sessions have visit_number > 1 "
      f"(lowest visit_number: {relabelled.visit_number.min()})")

96,809 of 96,809 relabeled sessions have visit_number > 1 (lowest visit_number: 2)


**Decision D-PR1:** journeys use the **arrival channel** (isTrueDirect → Direct). The last-click baseline keeps GA's labels, which is what the dashboards showed. A conservative relabel (only visits whose source reads `(direct)`) is kept for comparison; [notebook 05](05_analyze_attribution.ipynb) shows that form depends on the export day, not the visit.

## 4. D2 model table: corrected

External first-visit non-buyers. Label = purchase on a later visit within 30 days. Only first visits whose full 30-day window lies inside the data. Same train/test months as the lab.

In [8]:
rm = remarketing_table(sessions)
rm.to_parquet(ROOT / "data/processed/remarketing_table.parquet", index=False)
rm.groupby("split").agg(
    visitors=("buy_within_30d", "size"),
    buyers_30d=("buy_within_30d", "sum"),
    rate_pct=("buy_within_30d", lambda x: round(100 * x.mean(), 3)),
    revenue_30d_usd=("revenue_30d_usd", "sum"),
)

,visitors,buyers_30d,rate_pct,revenue_30d_usd
split,,,,
test,91431,323,0.353,53545.29
train,522599,1433,0.274,239574.97


In [9]:
first_visits = ext[ext.visit_number == 1].groupby("full_visitor_id").visit_id.nunique()
two = (first_visits > 1).sum()
print(f"{two:,} of {len(first_visits):,} external new visitors have two separate visits with visit_number = 1: "
      f"{two / len(first_visits):.2%} ({two / len(first_visits):.4%})")

608 of 668,583 external new visitors have two separate visits with visit_number = 1: 0.09% (0.0909%)


## 5. D1 journey tables

A purchase or 30 days of inactivity ends a journey. 30-day lookback. Journeys ending 2016-08-31 – 2017-07-01. Revenue capped per purchase session at the 99th percentile (USD 1,606, set with the key account included). Google employees and the key account ([`src/segments.py`](../src/segments.py)) are left out of the journeys.

In [10]:
t = touches(sessions)
orders = t.loc[t.purchased, "revenue_usd"]
cap = revenue_cap(sessions)
j = journeys(t, revenue_cap_usd=cap)
t.to_parquet(ROOT / "data/processed/touches.parquet", index=False)
j.to_parquet(ROOT / "data/processed/journeys.parquet", index=False)

conv = j[j.converted]
assert len(conv) == len(orders), "every external purchase in the period must end exactly one journey"
pd.Series({
    "journeys": len(j),
    "converting journeys": len(conv),
    "converting journeys with >1 touch": f"{(conv.n_touches > 1).mean():.1%}",
    "arrival path differs from GA labels": f"{(conv.arrival_path != conv.ga_path).mean():.1%}",
    "revenue cap (p99 purchase session)": f"${cap:,.0f}",
    "purchase sessions above the cap: share of revenue": f"{orders[orders > cap].sum() / orders.sum():.1%}",
    "revenue raw / capped": f"${conv.revenue_usd.sum():,.0f} / ${conv.revenue_capped_usd.sum():,.0f}",
})

journeys                                                          580759
converting journeys                                                 5058
converting journeys with >1 touch                                  42.4%
arrival path differs from GA labels                                29.6%
revenue cap (p99 purchase session)                                $1,606
purchase sessions above the cap: share of revenue                  17.2%
revenue raw / capped                                 $721,771 / $664,824
dtype: object

In [11]:
conv.arrival_path.value_counts().head(10).rename("converting journeys")

arrival_path
Organic Search                                1389
Direct                                        1203
Organic Search > Direct                        443
Direct > Direct                                440
Paid Search                                    205
Direct > Direct > Direct                       187
Organic Search > Direct > Direct               183
Direct > Direct > Direct > Direct              109
Organic Search > Direct > Direct > Direct       78
Direct > Direct > Direct > Direct > Direct      57
Name: converting journeys, dtype: int64

## 6. All-traffic figures for Ask and Prepare

These are the all-traffic figures that [reports/01_ask.md](../reports/01_ask.md) and [reports/02_prepare.md](../reports/02_prepare.md) quote; I compute them here because [notebook 01](01_prepare.ipynb) runs only small summary queries and doesn't load the clean table.

In [12]:
# Ask: channel profile, all traffic, GA's channel labels
profile = sessions.groupby("channel").agg(
    sessions=("session_key", "size"),
    purchase_sessions=("purchased", "sum"),
    revenue_usd=("revenue_usd", "sum"),
).sort_values("revenue_usd", ascending=False)
profile.loc["All traffic"] = profile.sum()
conv_rate = profile.purchase_sessions / profile.sessions
referral_vs_all = conv_rate["Referral"] / conv_rate["All traffic"]
social_share = profile.sessions["Social"] / profile.sessions["All traffic"]
display_per_purchase = profile.revenue_usd["Display"] / profile.purchase_sessions["Display"]
print(f"Referral converts at {referral_vs_all:.0f}x the all-traffic rate ({referral_vs_all:.2f}x)")
print(f"Social brings {social_share:.0%} of sessions ({social_share:.2%})")
print(f"Display earns ${display_per_purchase:,.0f} per purchase session (${display_per_purchase:,.2f})")
pd.DataFrame({
    "sessions": profile.sessions.map("{:,.0f}".format),
    "purchase sessions": profile.purchase_sessions.map("{:,.0f}".format),
    "conv. rate": conv_rate.map(lambda x: f"{x:.2%} ({x:.4%})"),
    "revenue": profile.revenue_usd.map(lambda x: f"${x:,.0f} (${x:,.2f})"),
})

Referral converts at 4x the all-traffic rate (3.97x)
Social brings 25% of sessions (25.02%)
Display earns $911 per purchase session ($911.44)


,sessions,purchase sessions,conv. rate,revenue
channel,,,,
Referral,"104,838","5,322",5.08% (5.0764%),"$717,600 ($717,600.25)"
Direct,"143,026","2,061",1.44% (1.4410%),"$498,530 ($498,530.03)"
Organic Search,"381,561","3,443",0.90% (0.9023%),"$377,076 ($377,075.81)"
Display,"6,262",143,2.28% (2.2836%),"$130,337 ($130,336.56)"
Paid Search,"25,326",469,1.85% (1.8519%),"$47,543 ($47,543.43)"
Social,"226,117",104,0.05% (0.0460%),"$8,397 ($8,396.78)"
Affiliates,"16,403",9,0.05% (0.0549%),$654 ($654.38)
(Other),120,1,0.83% (0.8333%),$12 ($11.99)
All traffic,"903,653","11,552",1.28% (1.2784%),"$1,780,149 ($1,780,149.23)"


In [13]:
# Ask: buyers and when they first buy. A visit split at midnight keeps one visit_id, so it counts once.
s = sessions.sort_values(["full_visitor_id", "visit_start_time"], kind="stable")
first_buy = s[s.purchased].groupby("full_visitor_id").agg(
    buy_vid=("visit_id", "first"), buy_ts=("visit_start_time", "first"))
m = s.join(first_buy, on="full_visitor_id", how="inner")
earlier = m.loc[(m.visit_start_time < m.buy_ts) & (m.visit_id != m.buy_vid), "full_visitor_id"].nunique()
v = s.groupby("full_visitor_id").agg(
    lowest_visit_number=("visit_number", "min"), first_visit_number=("visit_number", "first"),
    first_ts=("visit_start_time", "first"), first_vid=("visit_id", "first"), internal=("is_internal", "first"),
).join(first_buy)
print(f"visitors: {len(v):,}")
print(f"buyers (visitors with a purchase session): {len(first_buy):,}")
print(f"buyers with a visit before their first purchase: {earlier:,} = {earlier / len(first_buy):.0%} "
      f"({earlier / len(first_buy):.2%})")
print(f"visitors with no visit_number = 1 in the data (lowest visit_number > 1): {(v.lowest_visit_number > 1).sum():,}")
restart = (v.first_visit_number > 1) & (v.lowest_visit_number == 1)
print(f"visitors whose first visit in the data has visit_number > 1: {(v.first_visit_number > 1).sum():,} "
      f"({restart.sum():,} of them restart at visit_number = 1 on a later visit)")

# New visitors (lowest visit_number = 1) buying on the first visit vs a later one, as in Prepare query p08
new = v[v.lowest_visit_number == 1]
rows = {}
for name, nv in {"all traffic": new, "external": new[~new.internal]}.items():
    first = (nv.buy_vid == nv.first_vid).fillna(False)
    later = ((nv.buy_ts > nv.first_ts) & ~first).fillna(False)
    rows[name] = {
        "new visitors": f"{len(nv):,}",
        "bought on first visit": f"{first.sum():,}",
        "bought on a later visit": f"{later.sum():,}",
        "share bought later": f"{later.sum() / (first.sum() + later.sum()):.0%} "
                              f"({later.sum() / (first.sum() + later.sum()):.2%})",
        "first-visit buys after midnight (split visit)": f"{(first & (nv.buy_ts > nv.first_ts)).sum():,}",
    }
pd.DataFrame(rows).T

visitors: 714,167
buyers (visitors with a purchase session): 10,022
buyers with a visit before their first purchase: 5,210 = 52% (51.99%)
visitors with no visit_number = 1 in the data (lowest visit_number > 1): 12,544
visitors whose first visit in the data has visit_number > 1: 12,667 (123 of them restart at visit_number = 1 on a later visit)


,new visitors,bought on first visit,bought on a later visit,share bought later,first-visit buys after midnight (split visit)
all traffic,"701,623","4,449","4,637",51% (51.03%),13
external,"668,583","2,901","2,146",43% (42.52%),8


In [14]:
from src.segments import is_key_account

# Prepare §1: the Referral / (direct) segment and the flagged employees' Direct sessions
seg = sessions[(sessions.channel == "Referral") & (sessions.source == "(direct)")]
root = seg.is_internal_entry  # referral path "/", the internal rule (D-P1a)
seg_share = seg.revenue_usd.sum() / sessions.revenue_usd.sum()
other_rev = seg.loc[~root, "revenue_usd"].sum()
emp = sessions[sessions.is_internal]
first_entry = emp[emp.is_internal_entry].groupby("full_visitor_id").visit_start_time.min()
emp_direct = emp[(emp.channel == "Direct") & ~emp.is_internal_entry]
since_entry = emp_direct.visit_start_time - emp_direct.full_visitor_id.map(first_entry)
# Prepare §4: the two halves of a visit split at midnight share full_visitor_id and visit_id
split = sessions.groupby(["full_visitor_id", "visit_id"]).visit_start_time.agg(["size", "min", "max"]).query("size > 1")
split_gap = split["max"] - split["min"]
# Prepare §5: Display purchase sessions over $5k
display_sessions = sessions[sessions.channel == "Display"]
big = display_sessions[display_sessions.revenue_usd > 5000]
big_share = big.revenue_usd.sum() / display_sessions.revenue_usd.sum()
pd.Series({
    "Referral / (direct) sessions": f"{len(seg):,}",
    "  share of all revenue": f"{seg_share:.0%} ({seg_share:.2%})",
    "  sessions on path '/' (the internal rule)": f"{root.sum():,} = {root.mean():.1%} ({root.mean():.3%})",
    "  sessions on other paths": f"{(~root).sum():,}",
    "  revenue on other paths": f"${other_rev / 1000:.1f}k (${other_rev:,.2f})",
    "flagged employees' Direct sessions": f"{len(emp_direct):,}",
    "  before the visitor's first internal-link visit": f"{(since_entry < 0).sum():,}",
    "  after it": f"{(since_entry > 0).sum():,}",
    "  at the same second": f"{(since_entry == 0).sum():,}",
    "visits split at midnight": f"{(split['size'] == 2).sum():,} of {len(split):,} have exactly two rows",
    "  gap between the two start times": f"{split_gap.min():,} s to {split_gap.max() / 60:.0f} min "
                                         f"({split_gap.max():,} s = {split_gap.max() / 60:.1f} min)",
    "  median gap": f"{split_gap.median() / 60:.0f} min ({split_gap.median():,.0f} s)",
    "Display purchase sessions over $5k": f"{len(big):,}",
    "  visitors who made them": f"{big.full_visitor_id.nunique():,}",
    "  all made by the key account": bool(is_key_account(big.full_visitor_id).all()),
    "  made by flagged employees": f"{int(big.is_internal.sum()):,}",
    "  revenue, largest first": ", ".join(f"${x:,.0f}" for x in big.revenue_usd.sort_values(ascending=False)),
    "  share of Display revenue": f"{big_share:.1%} ({big_share:.3%})",
})

Referral / (direct) sessions                                                    68,933
  share of all revenue                                                    38% (38.13%)
  sessions on path '/' (the internal rule)                    67,236 = 97.5% (97.538%)
  sessions on other paths                                                        1,697
  revenue on other paths                                             $2.5k ($2,519.65)
flagged employees' Direct sessions                                               3,023
  before the visitor's first internal-link visit                                 2,798
  after it                                                                         225
  at the same second                                                                 0
visits split at midnight                              898 of 898 have exactly two rows
  gap between the two start times                   2 s to 92 min (5,546 s = 92.4 min)
  median gap                               